# RayNet · Colab runner

Run the cells **from the top, in order**. Details: `docs/06_COMPUTE.md`.

**Before the first run (once):** Colab → key icon (Secrets) → add `KAGGLE_API_TOKEN`
(or `KAGGLE_USERNAME` + `KAGGLE_KEY`) and turn on "Notebook access".

**Every session:** Runtime → Change runtime type → **GPU**.
Nothing is uploaded from your laptop: Colab downloads the data from Kaggle itself, keeps a copy
of the zip on Google Drive for the next sessions, and writes all run results to Drive.
When you are done: Runtime → Disconnect and delete runtime.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, sys, glob, shutil, subprocess, torch
print(sys.version)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "| CPUs:", os.cpu_count())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
!df -h /content | tail -1

## 2. Settings and Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/raynet"      # data zip copy, runs, checkpoints
COMMIT = "main"                               # or a commit hash; every run logs the exact hash
OFFICIAL_COMMIT = "fa3ef9b"                   # official baseline code (checked)
DATA = "/content/data/LunarRM"                # local disk: fast, needed for memory maps
WORKERS = min(8, os.cpu_count() or 2)
SLUG = "the-first-lunar-pathloss-radio-map-prediction-challenge"
for d in ("data", "runs", "official_runs"):
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)
print("workers:", WORKERS)

## 3. Our code (latest `main`, or the commit in `COMMIT`)

In [ ]:
if not os.path.exists("/content/raynet-lunar"):
    !git clone -q https://github.com/usarrahim/raynet-lunar.git /content/raynet-lunar
%cd /content/raynet-lunar
!git fetch -q origin && git checkout -q {COMMIT} && git pull -q --ff-only origin main 2>/dev/null; true
!git log --oneline -1
!pip install -q -e .

## 4. Data → local disk
1. Already on the local disk (same session): nothing to do.
2. Zip already on Drive (from an earlier session): copy it (fast) and unzip.
3. First time: download from Kaggle to the local disk, unzip, and keep the zip on Drive.

The first time takes longer (download about 10 GB + copy to Drive); later sessions only copy
from Drive.

In [ ]:
ZIP_DRIVE = f"{DRIVE}/data/LunarRM.zip"
ZIP_LOCAL = "/content/LunarRM.zip"

def data_root():
    hits = sorted(glob.glob(f"{DATA}/**/metadata.json", recursive=True))
    return os.path.dirname(hits[0]) if hits else None

def kaggle_login():
    from google.colab import userdata
    def secret(name):
        try:
            return userdata.get(name)
        except Exception:
            return None
    token, user, key = secret("KAGGLE_API_TOKEN"), secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
    elif user and key:
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
    else:
        raise SystemExit("Add KAGGLE_API_TOKEN (or KAGGLE_USERNAME + KAGGLE_KEY) to Colab Secrets.")

if data_root():
    print("data already on the local disk")
else:
    if os.path.exists(ZIP_DRIVE):
        print("copying the zip from Drive ...")
        shutil.copy(ZIP_DRIVE, ZIP_LOCAL)
    else:
        print("first time: downloading from Kaggle ...")
        kaggle_login()
        !pip install -q --upgrade kaggle
        !kaggle competitions download -c {SLUG} -p /content
        shutil.move(f"/content/{SLUG}.zip", ZIP_LOCAL)
        print("saving a copy of the zip on Drive for the next sessions ...")
        shutil.copy(ZIP_LOCAL, ZIP_DRIVE)
    os.makedirs(DATA, exist_ok=True)
    !unzip -q -o {ZIP_LOCAL} -d {DATA}
    os.remove(ZIP_LOCAL)
DATA = data_root()
print("DATA =", DATA)
!ls -la {DATA}
!df -h /content | tail -1

In [ ]:
# data check on the full data (1-2 minutes); the last line must say RESULT: PASS
!python -m raynet.check_data --root {DATA} --no-fig | tail -12

## 5. P2.2 speed test (about 10 minutes of GPU)
E3 at 5.8 GHz, 300 training steps for width 24 and 32. The `maps/s` numbers decide the
budget in `configs/base.yaml`. **Paste the printed lines to Claude Code.**

In [ ]:
for w in (24, 32):
    !python -m raynet.train --config configs/e3_residual_58.yaml --data-root {DATA} --epochs 1 --limit-batches 300 --dev-max-batches 20 --width {w} --num-workers {WORKERS} --out /content/speed --run-name w{w} --log-every 0 | grep -E "params,|epoch 0|peak_gpu|samples_per_s|\"gpu\"" 

## 6. P1.6: quick official baseline (RadioUNet, 5.8 GHz, 10 epochs, stage 1)
Through our wrapper: the dev terrains are removed from training and used as the monitor set;
the real `val` split is not used. Checkpoints go to Drive every epoch. If the runtime stops,
run cells 1-4 and this section again: `--resume auto` continues from the last epoch.

In [ ]:
if not os.path.exists("/content/official"):
    !git clone -q https://github.com/anderspearson206/lunar-radiomap-challenge.git /content/official
!cd /content/official && git checkout -q {OFFICIAL_COMMIT} && git log --oneline -1
OUT = f"{DRIVE}/official_runs"
RUN = "P16_58_s0_firstU"
# smoke test first (3 batches, about 1 minute)
!python scripts/run_official.py --official /content/official -- --data-root {DATA} --band 58 --loss masked --phase firstU --epochs 1 --limit-batches 3 --num-workers {WORKERS} --out /content/smoke --run-name smoke --panels 0

In [ ]:
# the real run (GPU hours!)
!python scripts/run_official.py --official /content/official -- --data-root {DATA} --band 58 --loss masked --phase firstU --epochs 10 --seed 0 --num-workers {WORKERS} --out {OUT} --run-name {RUN} --resume auto
!cat {OUT}/{RUN}_history.csv

## 7. Our experiments (only after Claude Code has fixed `configs/base.yaml`)
Pick the config and the seed. The run folder is on Drive (`{DRIVE}/runs/<id>_<band>_s<seed>`),
`last.pt` is written every epoch, and `--resume auto` continues after a disconnect.
At the end the LAST checkpoint is evaluated on dev and on the frozen val split.

In [ ]:
CONFIG = "configs/e3_residual_58.yaml"   # e2_unet_58 / e3_residual_58 / e3_residual_415 / ...
SEED = 0
import yaml
cfg = yaml.safe_load(open("configs/base.yaml"))
assert "PROVISIONAL" not in open("configs/base.yaml").read(), "base.yaml is not final yet (P2.2)"
exp = yaml.safe_load(open(CONFIG))
RUN = f"{exp['id']}_{exp['band']}_s{SEED}"
print("run:", RUN)
# smoke test (3 batches), then the real run
!python -m raynet.train --config {CONFIG} --data-root {DATA} --seed {SEED} --epochs 1 --limit-batches 3 --dev-max-batches 3 --num-workers {WORKERS} --out /content/smoke --run-name smoke --log-every 0 | grep "epoch 0" 

In [ ]:
!python -m raynet.train --config {CONFIG} --data-root {DATA} --seed {SEED} --num-workers {WORKERS} --out {DRIVE}/runs --resume auto
!python -m raynet.evaluate --run {DRIVE}/runs/{RUN} --split dev --data-root {DATA}
!python -m raynet.evaluate --run {DRIVE}/runs/{RUN} --split val --data-root {DATA}

## 8. What is on Drive

In [ ]:
!ls -la {DRIVE}/runs {DRIVE}/official_runs
print("Copy the run folders WITHOUT *.pt files to runs/ on the laptop, or tell Claude Code to read them from Drive.")